# 06 · Marketing Mix Modeling (MMM) & Channel Attribution
## Measuring True Channel Contribution with Adstock and Diminishing Returns

> ⚠️ **All data is fully synthetic.** Generated via `src/data/synthetic_generator.py`. No real media spend or attribution data is used.

**Business question:** Which channels are actually driving paid subscriptions — and how should budget be reallocated to maximize iROAS?

**Why MMM matters for VBB:** Last-click attribution systematically overcredits conversion-stage channels (retargeting, branded search) and undercredits awareness channels (YouTube, Meta prospecting). MMM + incrementality testing corrects this, enabling smarter budget allocation — a prerequisite for VBB to work correctly.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from scipy.optimize import minimize

from src.data.synthetic_generator import generate_media_channels

plt.style.use('seaborn-v0_8-whitegrid')
print('Ready.')

In [ ]:
df_media = generate_media_channels(n_months=24, seed=42)
print(f'Media dataset: {len(df_media):,} channel-month observations')
print(f'Channels: {df_media["channel"].unique().tolist()}')
df_media.head(8)

## 1. Channel Spend Distribution

In [ ]:
spend_by_channel = df_media.groupby('channel')['spend_brl'].sum().sort_values(ascending=False)
conversions_by_channel = df_media.groupby('channel')['conversions'].sum()
cpa_by_channel = spend_by_channel / conversions_by_channel
roas_by_channel = df_media.groupby('channel').apply(lambda x: x['revenue_brl'].sum() / x['spend_brl'].sum())

summary = pd.DataFrame({
    'Total Spend (R$)': spend_by_channel.map('{:,.0f}'.format),
    'Share': spend_by_channel.div(spend_by_channel.sum()).map('{:.1%}'.format),
    'Total Conversions': conversions_by_channel.map('{:,.0f}'.format),
    'CPA (R$)': cpa_by_channel.map('{:.1f}'.format),
    'ROAS': roas_by_channel.map('{:.2f}x'.format),
})
print('Channel Performance Summary (24 months, synthetic):')
print(summary.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

palette = sns.color_palette('husl', n_colors=len(spend_by_channel))

# Spend distribution
wedges, texts, autotexts = axes[0].pie(
    spend_by_channel.values,
    labels=spend_by_channel.index,
    autopct='%1.1f%%',
    colors=palette,
    startangle=140
)
axes[0].set_title('Media Spend Distribution by Channel', fontweight='bold')

# ROAS comparison
roas_sorted = roas_by_channel.sort_values(ascending=True)
bar_colors = ['#e74c3c' if v < 1.5 else '#f39c12' if v < 2.5 else '#2ecc71' for v in roas_sorted]
axes[1].barh(roas_sorted.index, roas_sorted.values, color=bar_colors, alpha=0.85)
axes[1].axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='Break-even (ROAS=1.0)')
axes[1].set_title('ROAS by Channel (Last-Click Attribution)', fontweight='bold')
axes[1].set_xlabel('ROAS')
axes[1].legend()

plt.suptitle('Channel Performance — Before MMM Correction (Synthetic)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Adstock Transformation

Media effects carry over in time. A TV or YouTube impression today influences subscription decisions next month. Adstock captures this decay.

In [ ]:
def adstock(spend_series: np.ndarray, decay: float = 0.5) -> np.ndarray:
    """Apply geometric adstock transformation to a spend time series."""
    adstocked = np.zeros_like(spend_series, dtype=float)
    for t in range(len(spend_series)):
        if t == 0:
            adstocked[t] = spend_series[t]
        else:
            adstocked[t] = spend_series[t] + decay * adstocked[t - 1]
    return adstocked

# Apply adstock with channel-specific decay rates
decay_rates = {
    'Meta Ads': 0.5, 'Google Ads': 0.3, 'YouTube Ads': 0.6,
    'Email': 0.2, 'Organic': 0.1, 'Referral': 0.2, 'Affiliate': 0.3
}

pivot = df_media.pivot(index='month', columns='channel', values='spend_brl').fillna(0)
adstocked_pivot = pivot.copy()
for channel in pivot.columns:
    adstocked_pivot[channel] = adstock(pivot[channel].values, decay_rates.get(channel, 0.4))

print('Adstock applied. Sample (Meta Ads):')
comparison = pd.DataFrame({'Raw': pivot['Meta Ads'].values[:6], 'Adstocked': adstocked_pivot['Meta Ads'].values[:6]})
print(comparison.round(0))

## 3. OLS Marketing Mix Model

In [ ]:
# Build monthly conversion totals and OLS features
monthly_conversions = df_media.groupby('month')['conversions'].sum().values
X_mmm = add_constant(adstocked_pivot.values)
model_ols = OLS(monthly_conversions, X_mmm).fit()

print('MMM OLS Results:')
print(f'  R²:       {model_ols.rsquared:.4f}')
print(f'  Adj. R²:  {model_ols.rsquared_adj:.4f}')
print(f'  Obs:      {int(model_ols.nobs)}')
print()

coef_df = pd.DataFrame({
    'Channel': ['Intercept'] + list(adstocked_pivot.columns),
    'Coefficient': model_ols.params,
    'Std Error': model_ols.bse,
    'p-value': model_ols.pvalues,
}).set_index('Channel').round(4)
print(coef_df[coef_df.index != 'Intercept'].sort_values('Coefficient', ascending=False))

## 4. Incremental ROAS by Channel (MMM-corrected)

In [ ]:
# Estimate incremental conversions per channel using model coefficients
channels = list(adstocked_pivot.columns)
total_spend = {ch: df_media[df_media['channel']==ch]['spend_brl'].sum() for ch in channels}
avg_subscription_value = 92.0  # R$ average LTV (cross-plan)

iroas_data = []
for i, ch in enumerate(channels):
    coef = model_ols.params[i + 1]  # skip intercept
    total_inc_conversions = max(coef * adstocked_pivot[ch].sum(), 0)
    inc_revenue = total_inc_conversions * avg_subscription_value
    iroas = inc_revenue / max(total_spend[ch], 1)
    iroas_data.append({'Channel': ch, 'iROAS': round(iroas, 3), 'Incremental Subs': int(total_inc_conversions)})

iroas_df = pd.DataFrame(iroas_data).sort_values('iROAS', ascending=False)
print('Incremental ROAS (MMM-corrected):')
print(iroas_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
bar_colors = ['#2ecc71' if v > 1.5 else '#f39c12' if v > 0.8 else '#e74c3c'
              for v in iroas_df['iROAS']]
ax.barh(iroas_df['Channel'], iroas_df['iROAS'], color=bar_colors, alpha=0.85)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='iROAS = 1.0 (break-even)')
ax.set_title('Incremental ROAS by Channel (MMM-corrected, Synthetic)\nGreen = profitable | Yellow = marginal | Red = below break-even',
             fontweight='bold')
ax.set_xlabel('Incremental ROAS')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Key Insight

**MMM reveals what last-click hides:**

Channels like branded Google search look efficient on last-click (low CPA, high conversion rate) but mostly capture demand generated by upper-funnel channels (YouTube, Meta prospecting). MMM separates true incremental contribution from attribution credit.

**The connection to VBB:**
Once we know which channels truly drive High-LTV subscriber acquisition, we can:
1. Reallocate budget toward incrementally efficient channels
2. Apply pLTV bidding within those channels
3. Measure the combined impact via geo-holdout or switchback experiment

MMM and VBB are not competing approaches — they operate at different levels of the same measurement stack: **MMM measures channel-level efficiency; VBB shifts the value composition within each channel.**